In [ ]:
import sys; sys.path.append('..')  # shared utils.py lives in the repo root
import torch
import pandas as pd
from pomegranate.hmm import DenseHMM
from pomegranate.distributions import Normal
from utils import log_returns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

spy = pd.read_csv('../data/spy_ohlcv.csv', index_col=0, parse_dates=True, skiprows=[1, 2])
log_returns = log_returns(spy['Close']).dropna()

# pomegranate expects (n_sequences, n_timesteps, n_features)
X = torch.tensor(log_returns.values, dtype=torch.float32).unsqueeze(0).unsqueeze(-1)

In [33]:
model = DenseHMM([Normal(), Normal()], max_iter=1000, tol=1e-4)
model.fit(X)

DenseHMM(
  (start): Silent()
  (end): Silent()
  (distributions): ModuleList(
    (0-1): 2 x Normal()
  )
)

In [34]:
hidden_states = model.predict(X)
hidden_states

tensor([[0, 0, 0,  ..., 0, 0, 0]])

In [35]:
print('Transition matrix:\n', model.edges.exp())
for i, d in enumerate(model.distributions):
    print(f'\nState {i}:  mean={d.means.item():.6f}  std={d.covs.item()**0.5:.6f}')

Transition matrix:
 tensor([[0.9852, 0.0147],
        [0.0302, 0.9698]])

State 0:  mean=0.000955  std=0.006731

State 1:  mean=-0.000780  std=0.018298


In [61]:
print(model.distributions[1].means)

Parameter containing:
tensor([-0.0018])


In [37]:
dur0 = 1 / (1 - model.edges.exp()[0, 0])
dur1 = 1 / (1 - model.edges.exp()[1, 1])

print(dur0)
print(dur1)

tensor(67.7662)
tensor(33.0966)


In [ ]:


eigenvalues, eigenvectors = np.linalg.eig(model.edges.exp().T)
idx = np.argmin(np.abs(eigenvalues - 1))
pi = np.real(eigenvectors[:, idx])
pi = pi / pi.sum()

print(pi)

tensor([0.6727, 0.3273])


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

#probs = model.predict_proba(X)[0]
dates = log_returns.index
emissions = model.forward(X)
log_probs = emissions - torch.logsumexp(emissions, dim=-1, keepdim=True)
probs = torch.exp(log_probs)

mask = (dates >= '2007-01-01') & (dates <= '2007-12-31')
dates_filtered = dates[mask]
probs_filtered = probs.squeeze(0)[mask]
price_filtered = spy['Close'].loc[dates_filtered]

fig, ax1 = plt.subplots(figsize=(14, 4))

# Probabilities on left axis
ax1.plot(dates_filtered, probs_filtered.numpy(), label=['panic', 'calm'])
ax1.set_ylabel('State probability')

# Price on right axis
ax2 = ax1.twinx()
ax2.plot(dates_filtered, price_filtered.values, color='black', alpha=0.4, linewidth=0.8, label='SPY price')
ax2.set_ylabel('Price')

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax1.xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.show()

In [ ]:

probs = model.predict_proba(X)[0]
dates = log_returns.index
mask = (dates >= '2007-01-01') & (dates <= '2007-12-31')

dates_filtered = dates[mask]
probs_filtered = probs.squeeze(0)[mask]
price_filtered = spy['Close'].loc[dates_filtered]

fig, ax1 = plt.subplots(figsize=(14, 4))

# Probabilities on left axis
ax1.plot(dates_filtered, probs_filtered.numpy(), label=['panic', 'calm'])
ax1.set_ylabel('State probability')

# Price on right axis
ax2 = ax1.twinx()
ax2.plot(dates_filtered, price_filtered.values, color='black', alpha=0.4, linewidth=0.8, label='SPY price')
ax2.set_ylabel('Price')

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax1.xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.show()

In [48]:
results = []
models = []

for n in range(2, 6):
    model = DenseHMM([Normal() for _ in range(n)], max_iter=1000, tol=1e-4)
    model.fit(X)
    logp = model.log_probability(X)
    k = n**2 + 2*n - 1
    bic = -2 * logp + k * np.log(len(X))
    results.append((n, logp, bic))
    models.append(model)

In [88]:
len(models)

4

In [ ]:
k = 3
model = models[k]

print(f"transition matrix: {models[k].edges.exp()}")

for i in range(len(models[k].distributions)):
    mean = model.distributions[i].means.item()
    std = model.distributions[i].covs.item() ** 0.5
    print(f"mean: {mean}, std: {std}, CV: {std / mean}")

eigenvalues, eigenvectors = np.linalg.eig(model.edges.exp().T)
idx = np.argmin(np.abs(eigenvalues - 1))
pi = np.real(eigenvectors[:, idx])
pi = pi / pi.sum()

print(f"stationary probs: {pi}")

#probs = model.predict_proba(X)[0]
dates = log_returns.index
emissions = model.forward(X)
log_probs = emissions - torch.logsumexp(emissions, dim=-1, keepdim=True)
probs = torch.exp(log_probs)

mask = (dates >= '2019-01-01') & (dates <= '2019-12-31')
dates_filtered = dates[mask]
probs_filtered = probs.squeeze(0)[mask]
price_filtered = spy['Close'].loc[dates_filtered]

fig, ax1 = plt.subplots(figsize=(14, 4))

# ax1.plot(dates_filtered, probs_filtered.numpy(), label = ['panic', 'calm', 'correction', 'bull'])
ax1.plot(dates_filtered, probs_filtered.numpy(), label = ['high-volatility', 'panic', 'calm', 'consolidation', 'bear'])
ax1.set_ylabel('State probability')

# Price on right axis
ax2 = ax1.twinx()
ax2.plot(dates_filtered, price_filtered.values, color='black', alpha=0.4, linewidth=0.8, label='SPY price')
ax2.set_ylabel('Price')

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax1.xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.show()